# 파이썬 머신러닝 완벽 가이드 4장

앙상블 학습, 랜덤 포레스트, GBM, XGBoost

## 03. 앙상블 학습

### 개념 정리

**앙상블 학습**은 여러 개의 분류기를 만들고 그 예측을 결합하여, 단일 분류기보다 신뢰도가 높은 최종 예측을 얻는 기법이다. 이미지·음성 같은 비정형 데이터에서는 딥러닝이 강하지만, 표 형태의 정형 데이터에서는 앙상블 계열이 대체로 가장 좋은 성능을 낸다.

| 유형 | 결합 대상 | 학습 데이터 | 대표 알고리즘 |
|---|---|---|---|
| 보팅(Voting) | 서로 다른 알고리즘 | 같은 데이터 세트 | VotingClassifier |
| 배깅(Bagging) | 같은 알고리즘(주로 결정 트리) | 부트스트랩 샘플(중복 허용) | 랜덤 포레스트 |
| 부스팅(Boosting) | 같은 알고리즘을 순차적으로 | 앞 모델이 틀린 데이터에 가중치 부여 | GBM, XGBoost, LightGBM |
| 스태킹(Stacking) | 여러 모델의 예측값 | 예측값을 다시 학습 데이터로 사용 | 메타 모델 |

- **부트스트래핑**: 원본 데이터에서 중복을 허용하여 무작위로 샘플링하는 방식이다. 교차 검증은 폴드끼리 데이터가 겹치지 않지만, 배깅은 겹침을 허용한다.

### 하드 보팅과 소프트 보팅

- **하드 보팅**: 각 분류기가 예측한 레이블 중 가장 많이 나온 레이블을 최종 결과로 정한다(다수결).
- **소프트 보팅**: 각 분류기가 낸 레이블별 확률을 평균하여 확률이 가장 높은 레이블을 고른다.
  - 예: 분류기 4개가 레이블 1에 대해 0.7, 0.2, 0.8, 0.9의 확률을 내면 평균은 0.65이고, 레이블 2의 평균은 0.35이므로 1로 결정한다.
- 확률 정보를 함께 쓰기 때문에 일반적으로 소프트 보팅의 성능이 더 좋아 많이 사용된다.

### 사이킷런 VotingClassifier

- `estimators`: `(이름, 분류기 객체)` 튜플의 리스트
- `voting`: `'hard'`(기본값) 또는 `'soft'`
- 보팅이 항상 개별 분류기보다 좋은 것은 아니다. 다만 여러 관점의 모델을 결합하면 처음 보는 데이터에 대한 유연성이 높아지는 경우가 많다.

In [ ]:
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

cancer = load_breast_cancer()

data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
data_df.head(3)

In [ ]:
# 로지스틱 회귀와 KNN을 소프트 보팅으로 결합
lr_clf = LogisticRegression(solver='liblinear')
knn_clf = KNeighborsClassifier(n_neighbors=8)

vo_clf = VotingClassifier(estimators=[('LR', lr_clf), ('KNN', knn_clf)], voting='soft')

X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target,
                                                    test_size=0.2, random_state=156)

vo_clf.fit(X_train, y_train)
pred = vo_clf.predict(X_test)
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

# 개별 분류기 성능과 비교
classifiers = [lr_clf, knn_clf]
for classifier in classifiers:
    classifier.fit(X_train, y_train)
    pred = classifier.predict(X_test)
    class_name = classifier.__class__.__name__
    print('{0} 정확도: {1:.4f}'.format(class_name, accuracy_score(y_test, pred)))

## 04. 랜덤 포레스트

### 개념 정리

- 배깅의 대표 알고리즘으로, 기반 알고리즘은 **결정 트리**이다.
- 여러 결정 트리가 각자 부트스트랩 샘플(원본과 건수는 같지만 데이터가 중복 추출된 서브세트)로 학습하고, 최종 예측은 소프트 보팅으로 정한다.
- 배깅(Bagging)은 **B**ootstrap **agg**regat**ing**의 줄임말이다.
- 앙상블 중에서는 학습 속도가 빠르고 CPU 병렬 처리(`n_jobs=-1`)가 잘 되어, 기준 모델을 먼저 만들 때 자주 쓰인다.

### 주요 하이퍼 파라미터

| 파라미터 | 의미 |
|---|---|
| `n_estimators` | 결정 트리 개수. 늘릴수록 성능이 좋아질 수 있으나 무한정 오르지는 않고 학습 시간이 늘어난다 |
| `max_features` | 분할 시 참고할 피처 수. 랜덤 포레스트의 기본값은 `sqrt`(전체 피처 수의 제곱근)이다 |
| `max_depth`, `min_samples_leaf`, `min_samples_split` | 결정 트리와 같이 과적합을 제어하는 파라미터 |

- 트리 기반 앙상블은 하이퍼 파라미터가 많아 튜닝에 시간이 오래 걸리지만, 튜닝으로 얻는 성능 향상은 크지 않은 경우가 많다.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df=pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(), columns=['dup_cnt'])
    feature_dup_df=feature_dup_df.reset_index()
    new_feature_name_df=pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    new_feature_name_df['column_name']=new_feature_name_df[['column_name','dup_cnt']].apply(lambda x: x[0]+'_'+str(x[1]) if x[1]>0 else x[0], axis=1)
    new_feature_name_df=new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

def get_human_dataset():
    feature_name_df = pd.read_csv('/content/features.txt', sep=r'\s+',
                                  header=None, names=['column_index', 'column_name'])
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    X_train = pd.read_csv('/content/X_train.txt', sep=r'\s+', names=feature_name)
    X_test = pd.read_csv('/content/X_test.txt', sep=r'\s+', names=feature_name)
    y_train = pd.read_csv('/content/y_train.txt', sep=r'\s+', header=None, names=['action'])
    y_test = pd.read_csv('/content/y_test.txt', sep=r'\s+', header=None, names=['action'])
    return X_train, X_test, y_train, y_test

X_train,X_test,y_train,y_test=get_human_dataset()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_clf = RandomForestClassifier(random_state=0, max_depth=8)
rf_clf.fit(X_train, y_train)
pred = rf_clf.predict(X_test)
accuracy = accuracy_score(y_test, pred)
print('랜덤 포레스트 정확도: {0:.4f}'.format(accuracy))

### GridSearchCV로 하이퍼 파라미터 튜닝

시간을 줄이기 위해 `n_estimators=100`, `cv=2`로 고정하고 트리 깊이와 리프 관련 파라미터만 탐색한다.

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [8, 16, 24],
    'min_samples_leaf': [1, 6, 12],
    'min_samples_split': [2, 8, 16]
}

rf_clf = RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
grid_cv = GridSearchCV(rf_clf, param_grid=params, cv=2, n_jobs=-1)
grid_cv.fit(X_train, y_train)

print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
# 찾은 최적 파라미터로 다시 학습한 뒤 테스트 세트로 평가
rf_clf1 = RandomForestClassifier(n_estimators=100, min_samples_leaf=6, max_depth=16,
                                 min_samples_split=2, random_state=0)
rf_clf1.fit(X_train, y_train)
pred = rf_clf1.predict(X_test)
print('예측 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ftr_importances_values = rf_clf1.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index=X_train.columns)
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

plt.figure(figsize=(8, 6))
plt.title('Feature importances Top 20')
sns.barplot(x=ftr_top20, y=ftr_top20.index)
plt.show()

## 05. GBM(Gradient Boosting Machine)

### 개념 정리

**부스팅**은 약한 학습기(weak learner)를 순서대로 학습시키면서, 앞 학습기가 틀린 데이터에 더 신경 쓰도록 만들어 오류를 줄여 가는 방식이다.

- **AdaBoost**: 잘못 분류된 데이터의 가중치를 키워 다음 학습기가 그 데이터를 더 잘 맞히게 한다. 마지막에는 각 학습기에 서로 다른 가중치(예: 0.3, 0.5, 0.8)를 주어 결합한다.
- **GBM**: AdaBoost와 비슷하지만 가중치 업데이트에 **경사 하강법**을 사용한다. 잔차 `h(x) = y - F(x)`를 줄이는 방향으로 반복적으로 학습한다.
- 분류(`GradientBoostingClassifier`)와 회귀 모두 가능하다.
- 일반적으로 랜덤 포레스트보다 성능이 조금 높지만, 순차 학습이라 **병렬 처리가 되지 않아 매우 느리다**.

### 주요 하이퍼 파라미터

| 파라미터 | 기본값 | 의미 |
|---|---|---|
| `loss` | `'log_loss'`(구버전 `'deviance'`) | 경사 하강법에서 쓸 비용 함수 |
| `learning_rate` | 0.1 | 각 학습기가 오류를 보정하는 정도(0~1). 작으면 정교하지만 느리고, 크면 빠르지만 최소 오류를 지나칠 수 있다 |
| `n_estimators` | 100 | 약한 학습기 개수 |
| `subsample` | 1 | 각 학습기가 사용할 데이터 비율. 과적합이 우려되면 1보다 작게 둔다 |

- `learning_rate`를 낮추고 `n_estimators`를 높이는 조합이 성능 향상에 도움이 되지만, 학습 시간이 크게 늘어난다.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
import time
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = get_human_dataset()

# 학습 시간 측정 시작
start_time = time.time()

gb_clf = GradientBoostingClassifier(random_state=0)
gb_clf.fit(X_train, y_train)
gb_pred = gb_clf.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)

print('GBM 정확도: {0:.4f}'.format(gb_accuracy))
print('GBM 수행 시간: {0:.1f} 초'.format(time.time() - start_time))

## 06. XGBoost(eXtra Gradient Boost)

### 개념 정리

GBM을 기반으로 하되 GBM의 약점(느린 속도, 과적합 규제 부재)을 보완한 알고리즘이다.

| 장점 | 내용 |
|---|---|
| 뛰어난 예측 성능 | 분류와 회귀 모두에서 성능이 좋다 |
| GBM 대비 빠른 속도 | 병렬 학습이 가능하다(랜덤 포레스트보다 빠르다는 뜻은 아니다) |
| 과적합 규제 | L1·L2 규제 등 자체 규제 기능이 있다 |
| 가지치기 | 더 이상 이득이 없는 분할을 제거해 분할 수를 줄인다 |
| 내장 교차 검증과 조기 중단 | 평가 지표가 더 좋아지지 않으면 반복을 중간에 멈춘다 |
| 결손값 처리 | 결측치를 자체적으로 처리한다 |

### 두 가지 사용 방식

- **파이썬 래퍼 XGBoost**: `xgboost` 고유 API이다. 데이터를 `DMatrix`로 바꾸고 `xgb.train()`으로 학습한다. `predict()`가 **확률**을 반환한다.
- **사이킷런 래퍼 XGBoost**: `XGBClassifier`, `XGBRegressor`이다. `fit()`/`predict()`를 쓰며 `GridSearchCV`, `Pipeline` 등과 함께 사용할 수 있다.

| 파이썬 래퍼 | 사이킷런 래퍼 |
|---|---|
| `eta` (기본 0.3) | `learning_rate` (기본 0.1) |
| `num_boost_round` | `n_estimators` |
| `sub_sample` | `subsample` |
| `lambda` | `reg_lambda` |
| `alpha` | `reg_alpha` |

### 주요 하이퍼 파라미터

- **일반 파라미터**: `booster`(gbtree/gblinear), `nthread` 등. 거의 바꾸지 않는다.
- **부스터 파라미터**
  - `eta`: 학습률. 보통 0.01~0.2를 쓴다.
  - `min_child_weight`: 가지를 나누는 데 필요한 가중치 합. 클수록 분할을 덜 한다.
  - `gamma`: 리프를 더 나눌지 결정하는 최소 손실 감소량. 클수록 과적합이 줄어든다.
  - `max_depth`(기본 6): 보통 3~10을 쓴다.
  - `sub_sample`, `colsample_bytree`: 데이터 행과 피처(열)의 샘플링 비율이다.
  - `lambda`(L2), `alpha`(L1): 규제 강도이다.
  - `scale_pos_weight`: 클래스 불균형을 보정한다.
- **학습 태스크 파라미터**
  - `objective`: `binary:logistic`(이진 분류), `multi:softmax`(다중 분류, `num_class` 필요), `multi:softprob`(다중 분류 확률)
  - `eval_metric`: `rmse`, `mae`, `logloss`, `error`, `merror`, `mlogloss`, `auc`

**과적합이 심할 때**: `eta`를 낮추고(대신 반복 횟수를 늘림), `max_depth`를 낮추고, `min_child_weight`와 `gamma`를 높이며, `subsample`과 `colsample_bytree`를 조정한다.

**조기 중단(early stopping)**: 예를 들어 `n_estimators=200`, `early_stopping_rounds=50`이면, 가장 좋은 검증 점수가 나온 뒤 50회 동안 개선이 없을 때 학습을 멈춘다. 조기 중단을 쓰려면 평가용 데이터 세트와 `eval_metric`이 반드시 필요하다.

In [ ]:
#pip install xgboost==1.5.0

In [ ]:
import xgboost

print(xgboost.__version__)

### 파이썬 래퍼 XGBoost: 위스콘신 유방암 예측

- 타깃: 악성(`malignant`)=0, 양성(`benign`)=1
- 데이터 분할: 전체의 80%를 학습용, 20%를 테스트용으로 나누고, 학습용을 다시 90%(최종 학습)와 10%(검증, 조기 중단용)로 나눈다.

In [ ]:
import xgboost as xgb
from xgboost import plot_importance
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()
features = dataset.data
labels = dataset.target

cancer_df = pd.DataFrame(data=features, columns=dataset.feature_names)
cancer_df['target'] = labels
cancer_df.head(3)

In [ ]:
print(dataset.target_names)
print(cancer_df['target'].value_counts())

In [ ]:
# 마지막 칼럼(target)을 레이블로, 나머지를 피처로 사용
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

# 학습 80% / 테스트 20%
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2, random_state=156)

# 학습 데이터를 다시 최종 학습 90% / 검증 10%로 분할
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=156)

print(X_train.shape, X_test.shape)
print(X_tr.shape, X_val.shape)

파이썬 래퍼는 전용 데이터 객체인 `DMatrix`를 사용한다. `data`에는 피처, `label`에는 레이블을 넣는다. 넘파이 배열과 DataFrame 모두 넣을 수 있다.

In [ ]:
dtr = xgb.DMatrix(data=X_tr, label=y_tr)
dval = xgb.DMatrix(data=X_val, label=y_val)
dtest = xgb.DMatrix(data=X_test, label=y_test)

In [ ]:
params = {'max_depth': 3,
          'eta': 0.05,
          'objective': 'binary:logistic',
          'eval_metric': 'logloss'
         }
num_rounds = 400

`xgb.train()`에 하이퍼 파라미터 딕셔너리와 `evals`(학습·평가 세트 목록), `early_stopping_rounds`를 넘긴다. 반복마다 `train-logloss`와 `eval-logloss`가 출력되고, 검증 logloss가 50회 동안 개선되지 않으면 400회를 다 채우기 전에 멈춘다.

In [ ]:
eval_list = [(dtr, 'train'), (dval, 'eval')]

xgb_model = xgb.train(params=params, dtrain=dtr, num_boost_round=num_rounds,
                      early_stopping_rounds=50, evals=eval_list)

In [ ]:
# 파이썬 래퍼의 predict()는 확률을 반환하므로 0.5를 기준으로 0/1을 정한다
pred_probs = xgb_model.predict(dtest)
print('predict() 수행 결괏값을 10개만 표시, 예측 확률 값으로 표시됨')
print(np.round(pred_probs[:10], 3))

preds = [1 if x > 0.5 else 0 for x in pred_probs]
print('예측값 10개만 표시:', preds[:10])

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def get_clf_eval(y_test, pred, pred_proba):
    confusion=confusion_matrix(y_test,pred)
    accuracy=accuracy_score(y_test, pred)
    precision=precision_score(y_test, pred)
    recall=recall_score(y_test,pred)
    f1=f1_score(y_test, pred)
    roc_auc=roc_auc_score(y_test,pred)
    print('오차행렬')
    print(confusion)
    print('정확도:{0:.4f},정밀도:{1:.4f},재현율:{2:.4f},F1:{3:.4f}, AUC:{4:.4f}'.format(accuracy,precision,recall,f1,roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_probs)

`plot_importance()`는 피처 중요도를 막대그래프로 바로 그려 준다. 기본 지표인 F 스코어는 해당 피처가 트리 분할에 사용된 횟수이다. 넘파이 배열로 학습하면 피처명이 `f0`, `f1`처럼 표시된다.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_model, ax=ax)

- `xgb.to_graphviz()`: 학습된 트리 구조를 시각화한다(Graphviz 설치 필요).
- `xgb.cv()`: 사이킷런의 `GridSearchCV`처럼 교차 검증을 수행하며 결과를 DataFrame으로 반환한다. 주요 인자는 `params`, `dtrain`, `num_boost_round`, `nfold`, `stratified`, `metrics`, `early_stopping_rounds`이다.

### 사이킷런 래퍼 XGBoost: XGBClassifier

검증 세트로 나누기 전의 `X_train`, `y_train` 전체로 학습한다.

In [ ]:
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=3, eval_metric='logloss')
xgb_wrapper.fit(X_train, y_train, verbose=True)
w_preds = xgb_wrapper.predict(X_test)
w_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test, w_preds, w_pred_proba)

위스콘신 데이터는 569건으로 작기 때문에, 검증 세트를 떼어 내면 학습 데이터가 줄어 오히려 성능이 낮아질 수 있다. 데이터가 충분히 많다면 검증 세트와 조기 중단을 함께 쓰는 편이 과적합 개선에 유리하다.

### 사이킷런 래퍼에서 조기 중단

- `eval_set=[(X_tr, y_tr), (X_val, y_val)]`처럼 지정하면 앞의 튜플은 `validation_0`(학습), 뒤의 튜플은 `validation_1`(검증)로 출력된다.
- XGBoost 2.0 이상에서는 `early_stopping_rounds`와 `eval_metric`을 `fit()`이 아니라 **생성자**에 넣어야 한다.

In [ ]:
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=3, early_stopping_rounds=50, eval_metric='logloss')
evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, eval_set=evals, verbose=True)
ws50_preds = xgb_wrapper.predict(X_test)
ws50_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test, ws50_preds, ws50_pred_proba)

`early_stopping_rounds`를 너무 작게 잡으면, 아직 성능이 좋아질 여지가 있는데도 학습이 일찍 끝나 예측 성능이 떨어질 수 있다. 10으로 줄여 비교해 본다.

In [ ]:
xgb_wrapper=XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=3, early_stopping_rounds=10, eval_metric='logloss')
xgb_wrapper.fit(X_tr,y_tr, eval_set=evals, verbose=True)
ws10_preds=xgb_wrapper.predict(X_test)
ws10_preds_proba=xgb_wrapper.predict_proba(X_test)[:,1]
get_clf_eval(y_test, ws10_preds, ws10_preds_proba)

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_wrapper, ax=ax)

## 정리

| 알고리즘 | 방식 | 장점 | 단점 |
|---|---|---|---|
| 보팅 | 서로 다른 모델의 다수결 또는 확률 평균 | 구현이 간단하다 | 항상 개별 모델보다 좋은 것은 아니다 |
| 랜덤 포레스트 | 결정 트리 + 배깅 | 빠르고 병렬 처리가 가능하며 튜닝 부담이 적다 | 부스팅보다 성능이 약간 낮은 경우가 많다 |
| GBM | 경사 하강법 기반 순차 부스팅 | 성능이 좋다 | 병렬 처리가 되지 않아 매우 느리다 |
| XGBoost | 규제와 병렬화를 더한 GBM | 성능, 조기 중단, 과적합 규제 | 대용량에서는 여전히 학습 시간이 길다 |